### Atividade 03 Gemma Local + RFM: Chatbot de Análise de Clientes com IA Generativa

#### Passo 1 - Instalar biblioteca e testar conexão

In [1]:
import sys
!{sys.executable} -m pip install ollama --quiet

import ollama

MODELO = "gemma3:4b"

# Teste rápido
resposta = ollama.chat(
    model=MODELO,
    messages=[{
        "role": "user",
        "content": "Olá! Responda em português: o que é RFM em 3 linhas?"
    }]
)

print("🤖 Gemma 4B diz:")
print(resposta['message']['content'])


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


🤖 Gemma 4B diz:
RFM significa **Recência, Frequência e Montante**, sendo um modelo de análise de dados de clientes. Ele classifica os clientes com base em quando compraram algo recentemente (recência), com que frequência compram (frequência) e quanto gastaram no total (montante). Essa segmentação ajuda empresas a personalizarem estratégias de marketing e oferecerem promoções mais direcionadas para cada tipo de cliente.



#### Passo 2 - Testes

##### Análise dos segmentos RFM

In [2]:
prompt = """
Você é um analista de dados especialista em e-commerce brasileiro.
Analise os segmentos de clientes abaixo e dê recomendações de negócio em português:

Segmento Campeão:
- 2.801 clientes (3% do total)
- Compraram em média há 220 dias
- Frequência média: 2.11 compras
- Ticket médio: R$ 308,59
- Gera 5.6% da receita total

Segmento Novo/Recente:
- 16.084 clientes (17.2% do total)
- Compraram em média há 42 dias
- Frequência média: 1.0 compra
- Ticket médio: R$ 133,91
- Gera 14% da receita total

Segmento Em Risco:
- 32.185 clientes (34.5% do total)
- Compraram em média há 272 dias
- Frequência média: 1.0 compra
- Ticket médio: R$ 295,78
- Gera 61.7% da receita total

Segmento Inativo:
- 42.287 clientes (45.3% do total)
- Compraram em média há 287 dias
- Frequência média: 1.0 compra
- Ticket médio: R$ 68,21
- Gera 18.7% da receita total

Para cada segmento dê:
1. Perfil do cliente
2. Principal oportunidade
3. Ação de marketing recomendada
"""

resposta = ollama.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt}]
)

print("🤖 Análise do Gemma 4B:")
print("=" * 60)
print(resposta['message']['content'])

🤖 Análise do Gemma 4B:
## Análise e Recomendações de Negócio para E-commerce Brasileiro - Segmentação de Clientes

Prezado(a),

Com base na análise detalhada dos seus segmentos de clientes, apresento um relatório com perfis, oportunidades e recomendações de ação específicas. O objetivo é otimizar sua estratégia de e-commerce e direcionar esforços para o crescimento sustentável do seu negócio.

**1. Segmento Campeão:**

* **1. Perfil do Cliente:** Este grupo representa seus clientes mais leais e valiosos. São consumidores experientes, que demonstram forte identificação com a sua marca, com uma frequência de compra significativa (2.11 vezes por período) e um ticket médio considerável (R$ 308,59). A longo prazo, essa base é crucial para a saúde do seu negócio.
* **2. Principal Oportunidade:**  Aumentar o lifetime value (LTV) desses clientes, incentivando compras adicionais e aprofundando o relacionamento com a marca. Eles são um excelente público para programas de fidelidade e ofertas per

##### E-mails por segmento

In [3]:
segmentos_info = {
    "Campeão": {
        "dias": 220, "frequencia": 2.1, "ticket": 308.59
    },
    "Novo/Recente": {
        "dias": 42,  "frequencia": 1.0, "ticket": 133.91
    },
    "Em Risco": {
        "dias": 272, "frequencia": 1.0, "ticket": 295.78
    },
    "Inativo": {
        "dias": 287, "frequencia": 1.0, "ticket": 68.21
    }
}

for segmento, dados in segmentos_info.items():
    prompt_email = f"""
    Escreva um e-mail de marketing curto em português (máximo 5 linhas)
    para clientes do segmento "{segmento}" de um e-commerce brasileiro.
    
    Perfil:
    - Última compra: há {dados['dias']} dias
    - Número de compras: {dados['frequencia']}
    - Ticket médio: R$ {dados['ticket']}
    
    O e-mail deve ter assunto e mensagem persuasiva para o perfil.
    """

    resp = ollama.chat(
        model=MODELO,
        messages=[{"role": "user", "content": prompt_email}]
    )

    print(f"\n{'='*55}")
    print(f"E-MAIL PARA SEGMENTO: {segmento}")
    print(f"{'='*55}")
    print(resp['message']['content'])


E-MAIL PARA SEGMENTO: Campeão
**Assunto:** Seu Exclusivo: Destaque Premium te Espera! ✨

Olá [Nome do Cliente],

Notamos que você é um Campeão por aqui! Para agradecer sua fidelidade, preparamos ofertas exclusivas e produtos incríveis que só você merece.  Reverta em nossos clientes mais valiosos e aproveite nosso novo catálogo! 🚀
[Link para o site]

Atenciosamente,
Equipe [Nome do E-commerce]

E-MAIL PARA SEGMENTO: Novo/Recente
**Assunto:** Lembranças boas e novidades incríveis! ✨

**Mensagem:**

Olá [Nome do Cliente]!

Notamos que você ama as nossas peças e seus últimos looks foram incríveis! 😉  Preparamos um cupom de desconto especial para você voltar a brilhar: Use o código **BEMVINDO10** e ganhe 10% OFF na sua próxima compra.  Volte sempre! ❤️

Atenciosamente,
[Nome do E-commerce] 
[Link para o site]

---

**Observações:**

*   Substitua "[Nome do Cliente]" pelo nome real do cliente (se disponível).
*   A oferta de 10% é apenas um exemplo; ajuste conforme a sua estratégia.
*   Adi

#####  Chatbot interativo

In [4]:
def chatbot_rfm(pergunta):
    contexto = """
    Dados RFM do e-commerce Olist Brasil (93.357 clientes):
    - Campeão (3%): compram com frequência, alto ticket, geram 5.6% receita
    - Novo/Recente (17.2%): compraram recentemente (42 dias), 1 compra só
    - Em Risco (34.5%): alto ticket mas inativos há 272 dias, geram 61.7% receita
    - Inativo (45.3%): baixo ticket, inativos há 287 dias, geram 18.7% receita
    """

    resposta = ollama.chat(
        model=MODELO,
        messages=[{
            "role": "user",
            "content": f"Contexto: {contexto}\n\nPergunta: {pergunta}\n\nResponda em português."
        }]
    )
    return resposta['message']['content']


perguntas = [
    "Qual segmento deve ser prioridade para campanhas?",
    "Como aumentar a frequência de compra dos clientes?",
    "Vale investir em reativar os clientes inativos?"
]

for p in perguntas:
    print(f"\n❓ {p}")
    print("-" * 50)
    print(chatbot_rfm(p))


❓ Qual segmento deve ser prioridade para campanhas?
--------------------------------------------------
Considerando os dados do e-commerce Olist Brasil, o segmento que deve receber prioridade nas campanhas é o **Em Risco** (34.5%).

**Justificativa:**

*   **Alto potencial de receita:** Apesar da inatividade prolongada, o segmento Em Risco possui o maior percentual de receita gerado (61.7%), indicando que eles têm a capacidade de comprar e gastar alto.
*   **Oportunidade de reengajamento:** A inatividade é reversível. Uma campanha focada em recuperar esses clientes com ofertas especiais, mensagens personalizadas ou lembretes de produtos que eles demonstraram interesse pode ser muito eficaz.

Embora os outros segmentos tenham seus méritos:

*   **Campeão:** São já lucrativos e não necessitam de campanhas intensivas.
*   **Novo/Recente:**  São um alvo fácil, mas com apenas uma compra, precisam de estímulo para fidelização.
*   **Inativo:** A inatividade é muito grande, tornando o esforç

#### Passo 3 - Conclusão

##### Informações do Modelo

| Característica | Descrição |
|---------------|-----------|
| Modelo utilizado | Google Gemma 3 4B (executado localmente via Ollama) |
| Tamanho | 3,3 GB |
| Execução | 100% local, sem acesso à internet e sem uso de API Key |
| Aplicação | Assistente de Análise RFM — Olist Brasil |

##### Atividades Realizadas pelo Gemma 3 4B

- Definiu corretamente os conceitos de **Recência, Frequência e Valor Monetário (RFM)**.
- Analisou os quatro segmentos de clientes com recomendações detalhadas de negócio.
- Gerou e-mails personalizados para cada segmento identificado.
- Respondeu perguntas estratégicas relacionadas aos resultados da análise.

##### Comparação entre Gemma 3 1B e Gemma 3 4B

| Modelo | Resultado |
|----------|-----------|
| Gemma 3 1B | Definiu RFM incorretamente e apresentou respostas mais superficiais |
| Gemma 3 4B | Definiu RFM corretamente e produziu análises mais detalhadas e consistentes |

##### Principais Insights Gerados pelo Modelo

- O segmento **Em Risco** foi identificado como prioridade para campanhas de recuperação, por concentrar aproximadamente **61,7% da receita**.
- Foram elaborados e-mails personalizados para cada perfil de cliente identificado na segmentação.
- O modelo sugeriu estratégias de reativação para clientes inativos, destacando oportunidades de recuperação de receita.
- Foram propostas ações de fidelização para clientes recentes e estratégias de retenção para clientes de maior valor.

##### Considerações Finais

O modelo **Google Gemma 3 4B** demonstrou capacidade de interpretar resultados de segmentação RFM, gerar recomendações de negócio e auxiliar na elaboração de estratégias de marketing. A execução local por meio do Ollama permitiu realizar toda a atividade sem dependência de serviços externos, evidenciando o potencial de modelos de linguagem de código aberto para apoiar análises de dados e tomada de decisão.